# Live Headline Sentiment Polling

## Integration Guide: Raw Per-Headline Sentiment

This notebook demonstrates how to integrate the **Permutable AI Live Headline Feed API** into a systematic trading workflow. By the end you will have a working polling loop that accumulates live headline sentiment into a local database and exposes it as a strategy signal.

### What the Endpoint Returns

`GET /v1/headlines/feed/live/ticker/{ticker}` returns the latest news headlines for a given ticker enriched with NLP-derived sentiment scores. Each record maps to the `AssetHeadline` schema:

| Field | Type | Description |
|---|---|---|
| `ticker` | `str` | Asset ticker (e.g. `BTC_CRY`) |
| `ticker_name` | `str` | Human-readable asset name |
| `publication_time` | `datetime` | When the headline was published |
| `topic_name` | `str` | Sentiment topic (e.g. `Macroeconomic Policy`) |
| `sentiment_score` | `float` | Continuous score from -1 (bearish) to +1 (bullish) |
| `bearish_probability` | `float` | Model probability of bearish sentiment |
| `neutral_probability` | `float` | Model probability of neutral sentiment |
| `bullish_probability` | `float` | Model probability of bullish sentiment |
| `topic_probability` | `float` | Model confidence the headline belongs to this topic |
| `match_type` | `str` | `EXPLICIT` / `IMPLICIT` / `COMBINED` |
| `language` | `str` | Language code (e.g. `en`) |
| `countries` | `list[str]` | Source countries |

### Polling Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│  Every 15 minutes                                               │
│                                                                 │
│  [API] ──► fetch_live_headlines(ticker)                         │
│                  │                                              │
│                  ▼                                              │
│           upsert_headlines()  ──► [SQLite: headline_sentiment]  │
│                                          │                      │
│                                          ▼                      │
│                               load from DB ──► strategy signals │
└─────────────────────────────────────────────────────────────────┘
```

**Key design decision — upsert not append:** The live endpoint always returns the most recent 2-hour window. Polling overlaps, so each poll will return headlines already seen in the previous poll. Upserting on `(ticker, publication_time, topic_name)` ensures the database stays deduplicated without any additional bookkeeping.

### Pipeline Steps

1. **Install** dependencies and configure API credentials
2. **Define** the fetch and upsert functions
3. **Dry-run** a single poll to validate connectivity and inspect raw data
4. **Visualise** initial data quality (sentiment distribution, topic breakdown)
5. **Run** the 15-minute polling loop
6. **Visualise** the accumulated live window (time series, heatmap)
7. **Derive** a threshold-based strategy signal from the stored data

### Prerequisites

- A Permutable AI API key (available from your account dashboard)
- Python 3.9+ with the packages listed in the cell below

In [ ]:
# Install required packages — run this cell once, then restart the kernel if needed
%pip install requests pandas matplotlib seaborn

In [ ]:
import getpass
import sqlite3
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})

## Configuration

The key is read via getpass so it is never displayed or stored in the notebook

In [ ]:
API_KEY = getpass.getpass("Enter your Permutable AI API key: ")

# ── Endpoint configuration ─────────────────────────────────────────────────────
BASE_URL = "https://copilot-api.permutable.ai/v1"

# Edit this list to match the tickers included in your licence
TICKERS = ["BTC_CRY", "ETH_CRY", "BZ_COM", "EUR_IND"]

# Request filters — adjust to your use case
MATCH_TYPE = "COMBINED"  # EXPLICIT | IMPLICIT | COMBINED
TOPIC_PRESET = "ALL"  # topic preset name or "ALL"
LANGUAGE_PRESET = "ALL"  # "ALL" | "EN" | language code
SOURCE_PRESET = "ALL"  # source preset name or "ALL"
SOURCE_COUNTRY_PRESET = "ALL"  # source country preset name or "ALL"
TOPIC_PROBABILITY_THRESHOLD = 0.1  # min topic confidence to include (0–1)
ABS_SENTIMENT_THRESHOLD = 0.1  # min |sentiment_score| to include (0–1)
LIMIT = 200  # headlines per request (1–1000)

# ── Polling interval ───────────────────────────────────────────────────────────
POLL_INTERVAL_SECONDS = 15 * 60  # poll every 15 minutes

# ── Local storage ──────────────────────────────────────────────────────────────
DB_PATH = Path("headline_sentiment.db")

print(f"Configured for {len(TICKERS)} tickers: {TICKERS}")
print(f"Polling interval : {POLL_INTERVAL_SECONDS // 60} minutes")
print(f"Local database   : {DB_PATH.resolve()}")

In [ ]:
def fetch_live_headlines(ticker: str) -> list[dict]:
    """
    Fetch live per-headline sentiment for a single ticker.

    Calls GET /v1/headlines/feed/live/ticker/{ticker} and returns a flat list
    of record dicts matching the AssetHeadline schema. The 'countries' list is
    flattened to a pipe-delimited string for SQLite compatibility.
    """
    url = f"{BASE_URL}/headlines/feed/live/ticker/{ticker}"
    params = {
        "match_type": MATCH_TYPE,
        "topic_preset": TOPIC_PRESET,
        "language_preset": LANGUAGE_PRESET,
        "source_preset": SOURCE_PRESET,
        "source_country_preset": SOURCE_COUNTRY_PRESET,
        "topic_probability_threshold": TOPIC_PROBABILITY_THRESHOLD,
        "abs_sentiment_threshold": ABS_SENTIMENT_THRESHOLD,
        "limit": LIMIT,
    }
    headers = {"x-api-key": API_KEY}

    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    records = response.json()
    for r in records:
        r["countries"] = "|".join(r.get("countries") or [])
        r["fetched_at"] = datetime.now(timezone.utc).isoformat()
    return records


# Quick connectivity check
print(f"Testing connectivity for {TICKERS[0]}...")
test_records = fetch_live_headlines(TICKERS[0])
print(f"  OK — {len(test_records)} headlines returned")

In [ ]:
def get_connection() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def setup_database() -> None:
    """Create the headline_sentiment table and index if they do not already exist."""
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS headline_sentiment (
                ticker               TEXT NOT NULL,
                publication_time     TEXT NOT NULL,
                topic_name           TEXT NOT NULL,
                ticker_name          TEXT,
                sentiment_score      REAL,
                bearish_probability  REAL,
                neutral_probability  REAL,
                bullish_probability  REAL,
                topic_probability    REAL,
                match_type           TEXT,
                language             TEXT,
                countries            TEXT,
                fetched_at           TEXT,
                PRIMARY KEY (ticker, publication_time, topic_name)
            )
        """)
        conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_hs_ticker_time
            ON headline_sentiment (ticker, publication_time)
        """)
    print(f"Database ready: {DB_PATH.resolve()}")


setup_database()

In [ ]:
def upsert_headlines(records: list[dict]) -> int:
    """
    Upsert headline records into the local SQLite database.

    INSERT OR REPLACE ensures that re-polling the same 2-hour window does not
    create duplicate rows. Records are uniquely identified by the composite key
    (ticker, publication_time, topic_name).

    Returns the number of rows written.
    """
    if not records:
        return 0

    columns = [
        "ticker",
        "publication_time",
        "topic_name",
        "ticker_name",
        "sentiment_score",
        "bearish_probability",
        "neutral_probability",
        "bullish_probability",
        "topic_probability",
        "match_type",
        "language",
        "countries",
        "fetched_at",
    ]
    placeholders = ", ".join(["?"] * len(columns))
    sql = f"INSERT OR REPLACE INTO headline_sentiment ({', '.join(columns)}) " f"VALUES ({placeholders})"
    rows = [tuple(r.get(c) for c in columns) for r in records]

    with get_connection() as conn:
        conn.executemany(sql, rows)
    return len(rows)


print("upsert_headlines() defined")

## Dry Run

Fetch and store headlines for all configured tickers once. Use this to verify your API key and inspect the raw schema before starting the polling loop.

In [ ]:
print("Running dry-run poll for all tickers...\n")
dry_run_rows = []

for ticker in TICKERS:
    try:
        records = fetch_live_headlines(ticker)
        n_written = upsert_headlines(records)
        dry_run_rows.append({"ticker": ticker, "headlines_fetched": len(records), "rows_written": n_written})
        print(f"  {ticker:12s}: {len(records):4d} headlines fetched | {n_written:4d} rows written")
    except requests.HTTPError as e:
        print(f"  {ticker}: HTTP {e.response.status_code} — {e.response.text[:200]}")
    except Exception as e:
        print(f"  {ticker}: {type(e).__name__}: {e}")

print()
display(pd.DataFrame(dry_run_rows))

# Preview the stored data
with get_connection() as conn:
    sample = pd.read_sql(
        "SELECT * FROM headline_sentiment ORDER BY publication_time DESC LIMIT 10",
        conn,
    )
print(f"\nSample rows from database:")
display(sample)

## Initial Visualisation

Inspect the sentiment score distribution and topic breakdown from the first poll.

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM headline_sentiment", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)

if df.empty:
    print("No data yet — run the dry-run cell above first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # ── Sentiment score distribution (KDE) per ticker ──────────────────────
    ax = axes[0]
    for ticker in df["ticker"].unique():
        scores = df.loc[df["ticker"] == ticker, "sentiment_score"].dropna()
        if not scores.empty:
            sns.kdeplot(scores, ax=ax, label=ticker, fill=True, alpha=0.20, linewidth=1.5)
    ax.axvline(0, color="black", linewidth=0.9, linestyle="--", label="Neutral")
    ax.set_title("Sentiment Score Distribution by Ticker", fontsize=11, fontweight="bold")
    ax.set_xlabel("Sentiment Score  (−1 bearish → +1 bullish)")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

    # ── Bull / Neutral / Bear probability stacked bar by topic (top 10) ───
    ax = axes[1]
    top_topics = df.groupby("topic_name")["sentiment_score"].count().nlargest(10).index
    topic_df = (
        df[df["topic_name"].isin(top_topics)]
        .groupby("topic_name")[["bullish_probability", "neutral_probability", "bearish_probability"]]
        .mean()
        .sort_values("bullish_probability", ascending=True)
    )
    topic_df.plot(
        kind="barh",
        stacked=True,
        ax=ax,
        color=["#2ecc71", "#95a5a6", "#e74c3c"],
        width=0.7,
        legend=False,
    )
    ax.legend(["Bullish", "Neutral", "Bearish"], fontsize=8, loc="lower right")
    ax.set_title("Mean Bull / Neutral / Bear Probability by Topic (Top 10)", fontsize=11, fontweight="bold")
    ax.set_xlabel("Probability")
    ax.set_xlim(0, 1)

    plt.suptitle(
        f"Initial Data Quality Check  —  {df['publication_time'].max().strftime('%Y-%m-%d %H:%M UTC')}",
        fontsize=12,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

    print(f"Total rows in database : {len(df)}")
    print(df.groupby("ticker")["sentiment_score"].describe().round(4))

## 15-Minute Polling Loop

This cell runs indefinitely, polling all configured tickers every 15 minutes and upserting results into the local SQLite database.

**Stop:** press the ■ (Interrupt Kernel) button or use **Kernel → Interrupt**.

In production this logic would run as a scheduled job (cron, Airflow, or a Docker Compose service). This notebook form is suitable for research and supervised live monitoring.

In [ ]:
poll_count = 0
print(f"Starting polling loop — {len(TICKERS)} tickers every {POLL_INTERVAL_SECONDS // 60} min")
print("Interrupt the kernel to stop gracefully.\n")

try:
    while True:
        poll_count += 1
        t_start = datetime.now(timezone.utc)
        print(f"[Poll {poll_count}]  {t_start.strftime('%Y-%m-%d %H:%M:%S UTC')}")

        for ticker in TICKERS:
            try:
                records = fetch_live_headlines(ticker)
                n = upsert_headlines(records)
                print(f"  {ticker:12s}: {len(records):4d} fetched | {n:4d} upserted")
            except requests.HTTPError as e:
                print(f"  {ticker}: HTTP {e.response.status_code} — skipping, will retry next poll")
            except Exception as e:
                print(f"  {ticker}: {type(e).__name__}: {e}")

        elapsed = (datetime.now(timezone.utc) - t_start).total_seconds()
        wait = max(0.0, POLL_INTERVAL_SECONDS - elapsed)
        print(f"  Completed in {elapsed:.1f}s.  Next poll in {wait / 60:.1f} min.\n")
        time.sleep(wait)

except KeyboardInterrupt:
    print(f"\nPolling stopped after {poll_count} poll(s).")
    print(f"All data saved to: {DB_PATH.resolve()}")

## Post-Poll Visualisation

Run this cell after one or more polling cycles to explore the accumulated live window:

1. Sentiment score over time per ticker (individual headlines + 15-min rolling mean)
2. Mean sentiment score per topic — one panel per ticker

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM headline_sentiment", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)
df = df.sort_values("publication_time")

if df.empty:
    print("No data yet — run the polling loop above first.")
else:
    active_tickers = [t for t in TICKERS if t in df["ticker"].unique()]

    # ── Plot 1: Sentiment time series per ticker ───────────────────────────
    fig, axes = plt.subplots(
        len(active_tickers),
        1,
        figsize=(14, 3.2 * len(active_tickers)),
        sharex=True,
    )
    if len(active_tickers) == 1:
        axes = [axes]

    for ax, ticker in zip(axes, active_tickers):
        sub = df[df["ticker"] == ticker].set_index("publication_time")["sentiment_score"].dropna()
        if sub.empty:
            ax.set_title(f"{ticker}  — no data")
            continue
        ax.scatter(sub.index, sub.values, alpha=0.30, s=14, color="#3498db", label="Headlines")
        rolling = sub.resample("15min").mean().rolling(4, min_periods=1).mean()
        ax.plot(rolling.index, rolling.values, color="#e74c3c", linewidth=1.8, label="15-min rolling mean")
        ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
        ax.set_ylabel("Sentiment", fontsize=9)
        ax.set_title(ticker, fontsize=10, fontweight="bold")
        ax.legend(fontsize=8, loc="upper left")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    plt.suptitle("Live Sentiment Time Series  —  2-Hour Window", fontsize=13, fontweight="bold")
    plt.xlabel("Time (UTC)")
    plt.tight_layout()
    plt.show()

    # ── Plot 2: Mean sentiment score per topic — one panel per ticker ─────
    top_topics = df.groupby("topic_name")["sentiment_score"].count().nlargest(12).index
    topic_sentiment = (
        df[df["topic_name"].isin(top_topics)].groupby(["ticker", "topic_name"])["sentiment_score"].mean().reset_index()
    )

    if not topic_sentiment.empty:
        fig, axes = plt.subplots(
            len(active_tickers),
            1,
            figsize=(14, 4.5 * len(active_tickers)),
            constrained_layout=True,
        )
        if len(active_tickers) == 1:
            axes = [axes]

        for ax, ticker in zip(axes, active_tickers):
            sub = topic_sentiment[topic_sentiment["ticker"] == ticker].sort_values("sentiment_score", ascending=True)
            if sub.empty:
                ax.set_title(f"{ticker}  — no data")
                continue
            colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in sub["sentiment_score"]]
            ax.barh(sub["topic_name"], sub["sentiment_score"], color=colors, edgecolor="white", height=0.6)
            ax.axvline(0, color="black", linewidth=0.9, linestyle="--")
            ax.set_title(f"{ticker}  —  Mean Sentiment Score by Topic", fontsize=11, fontweight="bold")
            ax.set_xlabel("Mean Sentiment Score  (−1 bearish → +1 bullish)")
            ax.tick_params(axis="y", labelsize=8)

        plt.suptitle("Topic Sentiment Breakdown by Ticker", fontsize=13, fontweight="bold")
        plt.show()

## Strategy Usage Example

Demonstrates how to load accumulated sentiment data from the local database and derive a simple directional signal suitable for a live systematic strategy.

**Signal logic:** aggregate raw headlines into 15-minute bins and compute:

| Metric | Description |
|---|---|
| `sentiment_mean` | Average direction (bullish/bearish) |
| `conviction` | Net sentiment / total absolute sentiment ∈ [−1, +1] |
| `headline_count` | News flow (attention proxy) |

A 2-period (30-min) rolling mean of `sentiment_mean` is then thresholded to produce a **LONG / SHORT / FLAT** signal per ticker.

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM headline_sentiment", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)

# ── 1. Bin to 15-minute periods ────────────────────────────────────────────────
df_agg = (
    df.groupby(["ticker", pd.Grouper(key="publication_time", freq="15min")])
    .agg(
        sentiment_mean=("sentiment_score", "mean"),
        headline_count=("sentiment_score", "count"),
        conviction=("sentiment_score", lambda s: (s.sum() / s.abs().sum() if s.abs().sum() > 0 else 0.0)),
    )
    .reset_index()
    .sort_values(["ticker", "publication_time"])
)

# ── 2. Smooth with a 30-minute rolling mean ────────────────────────────────────
df_agg["signal_smooth"] = df_agg.groupby("ticker")["sentiment_mean"].transform(
    lambda x: x.rolling(2, min_periods=1).mean()
)

# ── 3. Threshold-based position signal ────────────────────────────────────────
BULLISH_THRESHOLD = 0.05  # above this → LONG
BEARISH_THRESHOLD = -0.05  # below this → SHORT

df_agg["signal"] = "FLAT"
df_agg.loc[df_agg["signal_smooth"] >= BULLISH_THRESHOLD, "signal"] = "LONG"
df_agg.loc[df_agg["signal_smooth"] <= BEARISH_THRESHOLD, "signal"] = "SHORT"

# ── 4. Latest signal per ticker ────────────────────────────────────────────────
latest = (
    df_agg.sort_values("publication_time")
    .groupby("ticker")
    .last()
    .reset_index()[["ticker", "publication_time", "sentiment_mean", "conviction", "headline_count", "signal"]]
)
print("Latest strategy signals:")
display(latest)

# ── 5. Signal history chart ────────────────────────────────────────────────────
if len(df_agg) > 1:
    fig, ax = plt.subplots(figsize=(14, 5))

    for ticker, grp in df_agg.groupby("ticker"):
        ax.plot(grp["publication_time"], grp["signal_smooth"], label=ticker, linewidth=1.8)

    ax.axhline(BULLISH_THRESHOLD, color="#2ecc71", linewidth=1.0, linestyle="--", label="Bullish threshold")
    ax.axhline(BEARISH_THRESHOLD, color="#e74c3c", linewidth=1.0, linestyle="--", label="Bearish threshold")
    ax.axhline(0, color="black", linewidth=0.6, linestyle=":")
    ax.fill_between(
        df_agg["publication_time"].unique(),
        BEARISH_THRESHOLD,
        BULLISH_THRESHOLD,
        alpha=0.06,
        color="grey",
        label="FLAT zone",
    )

    ax.set_title("30-Min Smoothed Sentiment Signal by Ticker", fontsize=13, fontweight="bold")
    ax.set_xlabel("Time (UTC)")
    ax.set_ylabel("Rolling Mean Sentiment")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## Historical Backfill

### Why You Need This

The live endpoint (`/v1/headlines/feed/live/ticker/{ticker}`) returns only the **most recent 2-hour window** (up to 1,000 records). As a result, the local database only contains data from when polling started — charts have no historical depth and are unsuitable for strategy calibration or internal dashboards.

The **historical endpoint** (`/v1/headlines/feed/historical/ticker/{ticker}`) solves this:

| Property | Live | Historical |
|---|---|---|
| Window | Last 2 hours | Up to **90 days** back |
| Pagination | None (single response) | Keyset via `next_token` |
| Response schema | `list[AssetHeadline]` | `PaginatedAssetHeadlinesResponse` (`data`, `has_more`, `next_token`) |

### Backfill Strategy

Because both endpoints write into the **same SQLite table** via `upsert_headlines()`, backfilled records automatically merge with live-polled records. No schema changes or duplicate records are introduced.

**Pagination pattern:** each response contains `has_more` and `next_token`. Pass `next_token` as a query parameter on the next request and repeat until `has_more` is `False`.

```
fetch page 1 (start_date) ──► upsert ──► has_more?
                                              │
                              Yes ◄───────────┘──► fetch page N+1 (next_token) ──► upsert
                              No  ──► done
```

In [ ]:
from datetime import timedelta

BACKFILL_DAYS = 7  # days of history to fetch; max 90


def fetch_historical_headlines(ticker: str, start_date, end_date=None) -> list[dict]:
    """
    Paginate through GET /v1/headlines/feed/historical/ticker/{ticker} and return
    all matching records as a flat list.

    Keyset pagination: each response includes 'has_more' and 'next_token'.
    We loop until has_more is False, passing next_token into every subsequent request.
    """
    url = f"{BASE_URL}/headlines/feed/historical/ticker/{ticker}"
    params = {
        "start_date": start_date.isoformat(),
        "match_type": MATCH_TYPE,
        "topic_preset": TOPIC_PRESET,
        "language_preset": LANGUAGE_PRESET,
        "source_preset": SOURCE_PRESET,
        "source_country_preset": SOURCE_COUNTRY_PRESET,
        "topic_probability_threshold": TOPIC_PROBABILITY_THRESHOLD,
        "abs_sentiment_threshold": ABS_SENTIMENT_THRESHOLD,
        "limit": 1000,
    }
    if end_date:
        params["end_date"] = end_date.isoformat()
    headers = {"x-api-key": API_KEY}

    all_records, page = [], 1
    while True:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        body = response.json()
        records = body["data"]
        for r in records:
            r["countries"] = "|".join(r.get("countries") or [])
            r["fetched_at"] = datetime.now(timezone.utc).isoformat()
        all_records.extend(records)
        print(f"    page {page:3d}: {len(records):5d} records  |  total so far: {len(all_records):6d}")
        if not body["has_more"]:
            break
        params["next_token"] = body["next_token"]
        page += 1
    return all_records


def backfill_all_tickers(days_back: int = BACKFILL_DAYS) -> None:
    """Fetch and upsert historical headlines for all configured tickers."""
    start = (datetime.utcnow() - timedelta(days=days_back)).date()
    print(f"Backfilling {len(TICKERS)} tickers from {start} ({days_back} days)\n")
    for ticker in TICKERS:
        print(f"  {ticker}")
        try:
            records = fetch_historical_headlines(ticker, start)
            n = upsert_headlines(records)
            print(f"    {n} rows upserted into headline_sentiment\n")
        except requests.HTTPError as e:
            print(f"    HTTP {e.response.status_code} — skipping\n")
        except Exception as e:
            print(f"    {type(e).__name__}: {e}\n")


backfill_all_tickers()

In [ ]:
# Load the full DB — now includes both backfilled history and live-polled data
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM headline_sentiment", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)
df["date"] = df["publication_time"].dt.date

if df.empty:
    print("No data — run the backfill cell above first.")
else:
    active_tickers = [t for t in TICKERS if t in df["ticker"].unique()]
    colors = sns.color_palette("muted", len(active_tickers))
    date_range = sorted(df["date"].unique())

    # ── Plot 1: 30-min smoothed sentiment per ticker (full historical window) ──
    # Each headline is binned into 30-minute periods; a 3-period (90-min) rolling
    # mean is then applied, matching the smoothing used in the live section.
    fig, axes = plt.subplots(
        len(active_tickers), 1,
        figsize=(16, 3.5 * len(active_tickers)),
        sharex=True,
    )
    if len(active_tickers) == 1:
        axes = [axes]

    for ax, (ticker, color) in zip(axes, zip(active_tickers, colors)):
        sub = (
            df[df["ticker"] == ticker]
            .set_index("publication_time")["sentiment_score"]
            .dropna()
            .sort_index()
        )
        if sub.empty:
            ax.set_title(f"{ticker}  — no data")
            continue
        binned  = sub.resample("30min").mean()
        smoothed = binned.rolling(3, min_periods=1).mean()
        ax.scatter(sub.index, sub.values, alpha=0.12, s=8, color=color, label="Headlines")
        ax.plot(smoothed.index, smoothed.values, color=color, linewidth=1.6,
                label="30-min rolling mean")
        ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
        # Shade weekends for context
        for d in date_range:
            import datetime as _dt
            if _dt.date.fromisoformat(str(d)).weekday() >= 5:
                ax.axvspan(
                    pd.Timestamp(str(d), tz="UTC"),
                    pd.Timestamp(str(d), tz="UTC") + pd.Timedelta(days=1),
                    alpha=0.06, color="grey",
                )
        ax.set_ylabel("Sentiment", fontsize=9)
        ax.set_title(ticker, fontsize=10, fontweight="bold")
        ax.legend(fontsize=8, loc="upper left")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        ax.xaxis.set_major_locator(mdates.DayLocator())

    plt.suptitle(
        f"30-Min Smoothed Sentiment  —  {min(date_range)} to {max(date_range)}",
        fontsize=13, fontweight="bold",
    )
    plt.xlabel("Date")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    # ── Plot 2: Daily headline count per ticker (news flow context) ───────────
    daily_count = (
        df.groupby(["ticker", "date"])["sentiment_score"]
        .count()
        .rename("headline_count")
        .reset_index()
    )

    n_dates = len(date_range)
    bar_width = 0.8 / max(len(active_tickers), 1)
    x = range(n_dates)

    fig, ax = plt.subplots(figsize=(14, 4))
    for i, (ticker, color) in enumerate(zip(active_tickers, colors)):
        sub = daily_count[daily_count["ticker"] == ticker].set_index("date")
        counts = [sub.loc[d, "headline_count"] if d in sub.index else 0 for d in date_range]
        offsets = [xi + i * bar_width - (len(active_tickers) - 1) * bar_width / 2 for xi in x]
        ax.bar(offsets, counts, width=bar_width * 0.9, label=ticker, color=color, alpha=0.85)
    ax.set_xticks(list(x))
    ax.set_xticklabels([str(d) for d in date_range], rotation=30, ha="right")
    ax.set_title("Daily Headline Count by Ticker  (News Flow)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Headline Count")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"Date range in DB : {min(date_range)} → {max(date_range)}  ({len(date_range)} days)")
    print(f"Total rows       : {len(df):,}")

## Next Steps: Production Deployment

This notebook is an **integration reference** — it covers the complete workflow from API authentication through to a strategy-ready signal in a single, runnable document.

For a production deployment, see the accompanying **Docker Compose reference application** (coming soon), which packages this workflow into:

- A standalone **polling service** with configurable intervals and ticker lists
- A **PostgreSQL** store replacing the local SQLite database for multi-process access and durability
- A lightweight **REST API** exposing the latest signals to downstream order management systems
- A **monitoring dashboard** for real-time observation of news flow and sentiment drift

### Related Resources

- [Permutable AI API Documentation](https://docs.permutable.ai)
- `notebooks/backtesting/headline_cross_ticker_signal_assessment.ipynb` — full backtesting pipeline for validating the headline sentiment signal on historical data
- `notebooks/live/index_sentiment_polling.ipynb` — equivalent guide using the pre-aggregated hourly sentiment index endpoint